In [68]:
import numpy as np
import pandas as pd
import kagglehub
import numpy.linalg as linalg
import matplotlib.pyplot as plt

# Download latest version
path = kagglehub.dataset_download("farkhod77/abalone-age-predict")

print("Path to dataset files:", path)

Path to dataset files: /Users/leukaye/.cache/kagglehub/datasets/farkhod77/abalone-age-predict/versions/1


In [69]:
#Data preparation

abaloneData=pd.read_csv('abalone.data.csv')
abaloneData['gender']=abaloneData['gender'].map({'M':1, 'I':0, 'F':-1}) #code genders into numbers
rings=np.array(abaloneData['Rings'])
originaldata=np.array(abaloneData.drop('Rings',axis=1))
mu=np.mean(originaldata)
sigma=np.std(originaldata)
data=(originaldata-mu)/sigma
X=np.column_stack((np.ones(np.shape(data)[0]),data))
Y=np.column_stack((np.ones(np.shape(originaldata)[0]),originaldata))
corrmat=np.corrcoef(data,rowvar=False)

In [70]:
#Closed Form Linear Regression
mat=np.dot(linalg.inv(np.dot(X.transpose(),X)),X.transpose())
beta=np.dot(mat,rings)
print(beta)

[ 6.66024896  0.03053369 -0.82825179  6.02918378  4.49571341  4.08054577
 -9.00214376 -3.98483505  3.77871472]


In [71]:
#Grad desc. Linear Regression Normalised
test=np.ones(np.shape(X)[1])
RSS = lambda test: (rings - X @ test).T @ (rings - X @ test)
Grad= lambda test: -2*X.T@(rings-X@test) 
learningRate=0.01
iterations=500000
rss=np.zeros(iterations)
for i in range(iterations):
    test-= 1/np.shape(X)[0]*Grad(test)*learningRate
    rss[i]=RSS(test)
print(test)

KeyboardInterrupt: 

In [79]:
#Ridge Regression

A=data
l=75
betaR=linalg.solve(A.T@A+l*np.identity(np.shape(A)[1]), A.T@(rings-np.mean(rings)))
beta0=np.mean(rings)-np.mean(data,axis=0)@betaR
beta2=np.append(beta0,betaR)

In [80]:
testData=pd.read_csv('testdata.csv')
testData['gender']=testData['gender'].map({'M':1, 'I':0, 'F':-1})
testArray=np.array(testData.drop('Rings',axis=1))
testRings=np.array(testData['Rings'])
normalisedArray=np.column_stack((np.ones(np.shape(testArray)[0]),(testArray-mu)/sigma))
OLSPred=normalisedArray@beta
RidgePred=normalisedArray@beta2
print(OLSPred,RidgePred,testRings)
MSE=lambda y,y_hat: 1/np.shape(y)[0]*np.sum((y-y_hat)**2)
lsmse=MSE(testRings,OLSPred)
ridgemse=MSE(testRings,RidgePred)
print(lsmse,ridgemse)

[11.94791744 10.05580328  4.93091618  7.20278386  7.33643612  8.62877936
  8.35900046  7.688286    8.12091307  8.1727021   9.46052207  7.9035603
  9.83251872 10.49334507 13.11521196  8.85299474 11.59710979  8.93684289
 11.60755132  8.9340491  11.36735186 11.17052631 10.36051157  9.89802516
 10.61490449 10.98107046 10.77119496 12.61990317 13.34305387  8.99026827
 13.20860219  6.1637716   6.57498728  6.50711502  6.60875327  6.43190104
  7.23945681  7.7110147   7.34110191  8.41453139  9.11023032  7.99000022
  8.95808884  9.66148869  8.4490457   9.0747385   9.07252218  8.50011531
  8.83274631  9.61239085  9.65041192 10.5619386   9.43314982  9.09683873
  9.15090259  8.61679675 10.56690393  9.50418884  9.28248229  9.37166143
 10.88787205  9.45205293  9.61765756  9.94254442 10.59477366  9.43912746
 10.34471748  8.49570355  8.94604178 10.7930272  10.71500835 11.18851028
 11.07591391 10.53926356  8.76016916 10.77213122  9.99131991 11.4343964
 10.84869402 12.01534755 12.77339988  9.95075824 11.5